# BERTの分類問題のファインチューニングと評価

3つのデータセットで、3エポック学習を10回繰り返して平均のスコアを計算するコード

In [1]:
eval_result_list = []

In [7]:
!python ./bert_finetune_mean_f1score.py --pretrained_model_path="alabnii/jmedroberta-base-sentencepiece" \
    --trained_result_path="./jmedroberta_livedoor" \
    --checkpoint_output_path="./output" \
    --train_tsv_path="../../resources/dataset/seq_cls/livedoor/ldcc_first_sentence_train.tsv" \
    --valid_tsv_path="../../resources/dataset/seq_cls/livedoor/ldcc_first_sentence_valid.tsv" \
    --num_epochs=3 --num_runs=10 --learning_rate="5e-05"

eval_result_list.append("./jmedroberta_livedoor")

!python ./bert_finetune_mean_f1score.py --pretrained_model_path="alabnii/jmedroberta-base-sentencepiece" \
    --trained_result_path="./jmedroberta_tmclinical" \
    --checkpoint_output_path="./output" \
    --train_tsv_path="../../resources/dataset/seq_cls/tm_clinical_dataset/tm_clinical_1_train.tsv" \
    --valid_tsv_path="../../resources/dataset/seq_cls/tm_clinical_dataset/tm_clinical_1_val.tsv" \
    --num_epochs=3 --num_runs=10 --learning_rate="5e-05"

eval_result_list.append("./jmedroberta_tmclinical")

!python ./bert_finetune_mean_f1score.py --pretrained_model_path="alabnii/jmedroberta-base-sentencepiece" \
    --trained_result_path="./jmedroberta_tmnormal" \
    --checkpoint_output_path="./output" \
    --train_tsv_path="../../resources/dataset/seq_cls/tm_normal_dataset/tm_normal_1_train.tsv" \
    --valid_tsv_path="../../resources/dataset/seq_cls/tm_normal_dataset/tm_normal_1_val.tsv" \
    --num_epochs=3 --num_runs=10 --learning_rate="5e-05" \

eval_result_list.append("./jmedroberta_tmnormal")

{'peachy': 0, 'kaden-channel': 1, 'dokujo-tsushin': 2, 'smax': 3, 'movie-enter': 4, 'sports-watch': 5, 'livedoor-homme': 6, 'it-life-hack': 7, 'topic-news': 8}
Map: 100%|████████████████████████| 5892/5892 [00:00<00:00, 12564.29 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at alabnii/jmedroberta-base-sentencepiece and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
{'loss': 0.8816, 'grad_norm': 17.18137550354004, 'learning_rate': 3.871551334237902e-05, 'epoch': 0.68}
{'loss': 0.4812, 'grad_norm': 3.857255220413208, 'learning_rate': 2.740841248303935e-05, 'epoch': 1.36}
{'loss': 0.3232, 'grad_norm': 0.281409353017807, 'learning_rate': 1.6101311623699685e-05, 'epoch': 2.04}
{'loss': 0.1354, 'grad_norm': 0.03137337788939476, 'learning_rate': 4.79421076

In [8]:
# 学習のあとの結果を集計する

import json
import os

eval_result_list = list(set(eval_result_list))

all_model_score_dict = {}
mean_only_score_dict = {}

for eval_result_dir in eval_result_list:
    with open(os.path.join(eval_result_dir, "test_result.json"), mode="r", encoding="utf-8") as fp:
        result = json.load(fp)
    f1micro_score_dict = {}
    f1macro_score_dict = {}
    f1acc_score_dict = {}
    model_name = eval_result_dir.replace("./", "")

    all_score_dict = {}
    mean_score_dict = {}

    run_count = len(result.keys())
    for run_num, score_dict in result.items():
        all_score_dict[run_num] = score_dict
        for score_type, detail_score in score_dict.items():
            if not score_type in mean_score_dict.keys():
                mean_score_dict[score_type] = {}
            for k1, score in detail_score.items():
                if k1 in score_type:
                    for k2, score_elements in score.items():
                        if not k2 in mean_score_dict[score_type].keys():
                            mean_score_dict[score_type][k2] = 0.0
                        if k2 == "confidence_interval":
                            continue
                        else:
                            mean_score_dict[score_type][k2] += (score_elements / run_count)
                else:
                    continue
                
    
    all_model_score_dict[model_name] = all_score_dict.copy()
    mean_only_score_dict[model_name] = mean_score_dict.copy()

print(all_model_score_dict)
print(mean_only_score_dict)

{'jmedroberta_tmnormal': {'run-0': {'f1_micro': {'f1': {'confidence_interval': [0.7868318122555411, 0.8259452411994785], 'standard_error': 0.00999433172654941, 'score': 0.8063885267275098}, 'total_time_in_seconds': 4.0121908485889435, 'samples_per_second': 382.3347537267565, 'latency_in_seconds': 0.002615509027763327}, 'f1_marco': {'f1': {'confidence_interval': [0.7369871555065931, 0.794645999267615], 'standard_error': 0.014846061724927024, 'score': 0.7653980063502094}, 'total_time_in_seconds': 4.015231566503644, 'samples_per_second': 382.0452132318152, 'latency_in_seconds': 0.0026174912428315804}, 'accuracy': {'accuracy': {'confidence_interval': [0.7848761408083442, 0.8246414602346805], 'standard_error': 0.010034219521420064, 'score': 0.8057366362451108}, 'total_time_in_seconds': 4.050378603860736, 'samples_per_second': 378.73002749368254, 'latency_in_seconds': 0.002640403261969189}}, 'run-1': {'f1_micro': {'f1': {'confidence_interval': [0.7900912646675359, 0.8292046936114733], 'stand

In [9]:
import time

with open(f"all_score_dict_{int(time.time())}.json", mode="w", encoding="utf-8") as fp:
    json.dump(all_model_score_dict, fp, ensure_ascii=False)

with open(f"mean_score_dict_{int(time.time())}.json", mode="w", encoding="utf-8") as fp:
    json.dump(mean_only_score_dict, fp, ensure_ascii=False)
